In [ ]:
# ============================================================
# Multi-Model DewPoint TRUE FORECASTING
# (Trained on Fourier-Deseasonalized Data)
# SARIMA, LSTM, PatchTST
# ============================================================
# Data: ~110 years of DAILY data with seasonal component removed
# Training: deseasonalized DewPoint up to 2011
# Forecasting: predict deseasonalized DewPoint for 2012-2021
#              then add seasonal component back for evaluation
# Evaluation: monthly averages vs actual DewPoint
# ============================================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Load daily data (with deseasonalized columns from reverse_transform.py)
df = pd.read_csv("df_linear_merged.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["DayOfYear"] = df["Date"].dt.dayofyear

print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Shape: {df.shape}")

# Train/test split
train_daily = df[df["Year"] <= 2011].copy()
test_daily = df[df["Year"] >= 2012].copy()

print(f"\nTrain: {len(train_daily)} days | Test: {len(test_daily)} days")

# Ground truth: actual monthly averages (original DewPoint)
true_monthly = test_daily.groupby(["Year", "Month"]).agg(
    DewPoint=("DewPoint", "mean")
).reset_index()
true_monthly_dew = true_monthly["DewPoint"].values
print(f"Test months: {len(true_monthly)}")
df.head()

In [ ]:
# Rebuild the seasonal model so we can add it back to predictions later
# This must match exactly what reverse_transform.py computed
from sklearn.linear_model import LinearRegression

sin1 = np.sin(2 * np.pi * df["DayOfYear"] / 365.25)
cos1 = np.cos(2 * np.pi * df["DayOfYear"] / 365.25)
sin2 = np.sin(4 * np.pi * df["DayOfYear"] / 365.25)
cos2 = np.cos(4 * np.pi * df["DayOfYear"] / 365.25)
fourier_X = np.column_stack([sin1, cos1, sin2, cos2])

seasonal_dew_model = LinearRegression()
seasonal_dew_model.fit(fourier_X, df["DewPoint"].values)

def get_seasonal_value(day_of_year):
    """Return the seasonal DewPoint component for a given day-of-year."""
    s1 = np.sin(2 * np.pi * day_of_year / 365.25)
    c1 = np.cos(2 * np.pi * day_of_year / 365.25)
    s2 = np.sin(4 * np.pi * day_of_year / 365.25)
    c2 = np.cos(4 * np.pi * day_of_year / 365.25)
    x = np.column_stack([s1, c1, s2, c2]) if hasattr(day_of_year, '__len__') else np.array([[s1, c1, s2, c2]])
    return seasonal_dew_model.predict(x)

# Verify: seasonal + deseasonal should reconstruct original
recon = get_seasonal_value(df["DayOfYear"].values).flatten() + df["DewPoint_deseasonal"].values
error = np.abs(recon - df["DewPoint"].values).max()
print(f"Reconstruction check: max error = {error:.10f} (should be ~0)")
print(f"Seasonal model intercept (mean DewPoint): {seasonal_dew_model.intercept_:.4f} °C")
print("Seasonal model ready — will be used to add back seasonal component to predictions.")

In [ ]:
# Helpers
all_results = {}

def daily_to_monthly(dates, values):
    """Aggregate daily predictions to monthly averages."""
    tmp = pd.DataFrame({"Date": dates, "pred": values})
    tmp["Year"] = tmp["Date"].dt.year
    tmp["Month"] = tmp["Date"].dt.month
    monthly = tmp.groupby(["Year", "Month"])["pred"].mean().reset_index()
    monthly = monthly.sort_values(["Year", "Month"])
    return monthly["pred"].values

def add_seasonal_back(dates, deseasonal_values):
    """Add seasonal component back to deseasonalized predictions."""
    doys = pd.to_datetime(dates).dayofyear
    seasonal = get_seasonal_value(doys.values).flatten()
    return deseasonal_values + seasonal

## Model 1: SARIMA
Univariate SARIMA on **monthly deseasonalized DewPoint**.
Since the seasonal component is already removed, we use a non-seasonal ARIMA.
After forecasting, add seasonal component back for evaluation.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Monthly deseasonalized DewPoint for SARIMA
train_monthly_sarima = train_daily.groupby(["Year", "Month"]).agg(
    DewPoint_deseasonal=("DewPoint_deseasonal", "mean"),
).reset_index()
train_monthly_sarima["Date"] = pd.to_datetime(
    train_monthly_sarima[["Year", "Month"]].assign(Day=1)
)
train_monthly_sarima = train_monthly_sarima.set_index("Date").sort_index()

# Since seasonality is removed, use simpler ARIMA (no seasonal component)
# But keep seasonal_order for any residual seasonality the Fourier didn't capture
print("Fitting SARIMAX on monthly deseasonalized DewPoint...")
sarima_model = SARIMAX(
    train_monthly_sarima["DewPoint_deseasonal"],
    order=(2, 0, 1),
    seasonal_order=(1, 0, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_fit = sarima_model.fit(disp=False, maxiter=500)
print(sarima_fit.summary().tables[0])

# Forecast deseasonalized monthly values
n_test_months = len(true_monthly)
sarima_pred_deseas = sarima_fit.forecast(steps=n_test_months).values

# Add seasonal component back: use mid-month day-of-year
test_month_dates = pd.to_datetime(true_monthly[["Year", "Month"]].assign(Day=15))
test_month_doys = test_month_dates.dt.dayofyear.values
sarima_pred_monthly = sarima_pred_deseas + get_seasonal_value(test_month_doys).flatten()

all_results["SARIMA"] = {"pred_monthly": sarima_pred_monthly, "true_monthly": true_monthly_dew}

sarima_mae = mean_absolute_error(true_monthly_dew, sarima_pred_monthly)
sarima_rmse = np.sqrt(mean_squared_error(true_monthly_dew, sarima_pred_monthly))
sarima_r2 = r2_score(true_monthly_dew, sarima_pred_monthly)
print(f"\nSARIMA Monthly DewPoint — MAE: {sarima_mae:.3f} | RMSE: {sarima_rmse:.3f} | R2: {sarima_r2:.4f}")

## Shared Data Preparation for Neural Network Models (LSTM & PatchTST)
Trained on **deseasonalized DewPoint** — the seasonal cycle is already removed.

**Features:** `[DewPoint_deseasonal_scaled, year_norm]` — just 2 features.
No doy encoding needed since the seasonal pattern has been subtracted.

**Test-time:** Autoregressive forecasting, then add seasonal component back.

In [ ]:
# Feature engineering — deseasonalized data
df_nn = df.copy()

# Year trend normalized to [0, 1]
year_min, year_max = df_nn["Year"].min(), df_nn["Year"].max()
df_nn["year_norm"] = (df_nn["Year"] - year_min) / (year_max - year_min)

train_mask = df_nn["Year"] <= 2011
train_only_mask = df_nn["Year"] <= 2001

# Scale deseasonalized DewPoint using training data only
scaler = StandardScaler()
scaler.fit(train_daily[["DewPoint_deseasonal"]])
df_nn["dew_deseas_scaled"] = scaler.transform(df_nn[["DewPoint_deseasonal"]])

WINDOW_SIZE = 365

# Features: deseasonalized DewPoint (scaled) + year trend
FEATURE_COLS = ["dew_deseas_scaled", "year_norm"]
NUM_FEATURES = len(FEATURE_COLS)
DEW_IDX = 0  # index of dew_deseas_scaled in FEATURE_COLS

class DailyClimateDataset(Dataset):
    """Sliding window dataset. Target: next day's deseasonalized DewPoint (scaled)."""
    def __init__(self, data, window_size=WINDOW_SIZE):
        self.data = torch.FloatTensor(data)
        self.window_size = window_size

    def __len__(self):
        return len(self.data) - self.window_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.window_size]
        y = self.data[idx + self.window_size, DEW_IDX]
        return x, y

all_data = df_nn[FEATURE_COLS].values
train_end = train_only_mask.sum()
val_end = train_mask.sum()

train_data = all_data[:train_end]
val_data = all_data[train_end - WINDOW_SIZE : val_end]

train_dataset = DailyClimateDataset(train_data)
val_dataset = DailyClimateDataset(val_data)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Initial window for autoregressive forecasting
initial_window = all_data[val_end - WINDOW_SIZE : val_end].copy()

# Test dates
test_dates_nn = df_nn.iloc[val_end:]["Date"].values
n_test_days = len(test_dates_nn)

print(f"Features: {FEATURE_COLS} ({NUM_FEATURES} features)")
print(f"Trained on DESEASONALIZED DewPoint (seasonal removed)")
print(f"Train samples: {len(train_dataset):,}, Val: {len(val_dataset):,}")
print(f"Test days to forecast: {n_test_days}")
print(f"Test date range: {test_dates_nn[0]} to {test_dates_nn[-1]}")

In [ ]:
def train_nn_model(model, train_loader, val_loader, train_dataset, val_dataset,
                   num_epochs=50, patience=8, lr=1e-3):
    """Shared training loop for NN models."""
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)
    criterion = nn.MSELoss()

    train_losses, val_losses = [], []
    best_val_loss = float("inf")
    patience_counter = 0
    best_state = {k: v.clone() for k, v in model.state_dict().items()}

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            if torch.isnan(loss):
                print(f"  WARNING: NaN loss at epoch {epoch+1}, skipping batch")
                continue
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item() * x.size(0)
        epoch_loss /= len(train_dataset)

        model.eval()
        epoch_val = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                loss = criterion(pred, y)
                epoch_val += loss.item() * x.size(0)
        epoch_val /= len(val_dataset)

        train_losses.append(epoch_loss)
        val_losses.append(epoch_val)
        scheduler.step()

        if not np.isnan(epoch_val) and epoch_val < best_val_loss:
            best_val_loss = epoch_val
            patience_counter = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or patience_counter == 0:
            print(f"  Epoch {epoch+1:3d}/{num_epochs} | Train: {epoch_loss:.6f} | Val: {epoch_val:.6f} {'*' if patience_counter==0 else ''}")

        if patience_counter >= patience:
            print(f"  Early stop at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    print(f"  Best val loss: {best_val_loss:.6f}")
    return model, train_losses, val_losses


def forecast_nn_autoregressive(model, initial_window, n_days, test_dates, scaler):
    """
    Autoregressive forecasting on deseasonalized DewPoint.
    Predicts one day's deseasonalized value, feeds it back.
    Then adds seasonal component back for final output.
    """
    model.eval()
    window = initial_window.copy()
    predictions_deseas_scaled = []
    
    dew_mean, dew_std = scaler.mean_[0], scaler.scale_[0]
    
    with torch.no_grad():
        for i in range(n_days):
            x = torch.FloatTensor(window).unsqueeze(0).to(device)
            pred_scaled = model(x).item()
            predictions_deseas_scaled.append(pred_scaled)
            
            # Known features for next day
            date = pd.Timestamp(test_dates[i])
            year_norm = (date.year - year_min) / (year_max - year_min)
            
            new_row = np.array([pred_scaled, year_norm])
            window = np.vstack([window[1:], new_row])
            
            if (i + 1) % 365 == 0:
                print(f"    Forecasted {i+1}/{n_days} days...")
    
    # Inverse scale to get deseasonalized predictions
    preds_deseas_scaled = np.array(predictions_deseas_scaled)
    preds_deseas = preds_deseas_scaled * dew_std + dew_mean
    
    # Add seasonal component back → actual DewPoint predictions
    preds_daily = add_seasonal_back(test_dates, preds_deseas)
    
    # Aggregate to monthly
    preds_monthly = daily_to_monthly(test_dates, preds_daily)
    
    return preds_daily, preds_monthly

print("Training and forecasting utilities ready.")

## Model 2: LSTM
2-layer LSTM trained on deseasonalized DewPoint.
Autoregressive forecasting, then seasonal component added back.

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim=NUM_FEATURES, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.head(last).squeeze(-1)

lstm_model = LSTMModel()
total_params = sum(p.numel() for p in lstm_model.parameters())
print(f"LSTM parameters: {total_params:,}")

print("\nTraining LSTM on deseasonalized DewPoint...")
lstm_model, lstm_train_losses, lstm_val_losses = train_nn_model(
    lstm_model, train_loader, val_loader, train_dataset, val_dataset,
    lr=3e-4
)

print("\nAutoregressive forecasting with LSTM...")
lstm_pred_daily, lstm_pred_monthly = forecast_nn_autoregressive(
    lstm_model, initial_window, n_test_days, test_dates_nn, scaler
)
all_results["LSTM"] = {"pred_monthly": lstm_pred_monthly, "true_monthly": true_monthly_dew}

lstm_mae = mean_absolute_error(true_monthly_dew, lstm_pred_monthly)
lstm_rmse = np.sqrt(mean_squared_error(true_monthly_dew, lstm_pred_monthly))
lstm_r2 = r2_score(true_monthly_dew, lstm_pred_monthly)
print(f"\nLSTM Monthly DewPoint — MAE: {lstm_mae:.3f} | RMSE: {lstm_rmse:.3f} | R2: {lstm_r2:.4f}")

## Model 3: PatchTST
Patch-based Transformer trained on deseasonalized DewPoint.
Autoregressive forecasting, then seasonal component added back.

In [ ]:
class PatchTST(nn.Module):
    """
    PatchTST for daily time series.
    365-day window -> 73 patches of 5 days each.
    """
    def __init__(self, input_dim=NUM_FEATURES, seq_len=WINDOW_SIZE,
                 patch_len=5, d_model=64, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.patch_len = patch_len
        self.num_patches = seq_len // patch_len
        self.d_model = d_model

        self.patch_proj = nn.Linear(patch_len * input_dim, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)

        self.head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        B = x.size(0)
        usable = self.num_patches * self.patch_len
        x = x[:, :usable, :]
        x = x.reshape(B, self.num_patches, self.patch_len * x.size(2))
        x = self.patch_proj(x) + self.pos_embed
        x = self.transformer(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.head(x).squeeze(-1)

patchtst_model = PatchTST()
total_params = sum(p.numel() for p in patchtst_model.parameters())
print(f"PatchTST parameters: {total_params:,}")

print("\nTraining PatchTST on deseasonalized DewPoint...")
patchtst_model, ptst_train_losses, ptst_val_losses = train_nn_model(
    patchtst_model, train_loader, val_loader, train_dataset, val_dataset,
    lr=3e-4
)

print("\nAutoregressive forecasting with PatchTST...")
ptst_pred_daily, ptst_pred_monthly = forecast_nn_autoregressive(
    patchtst_model, initial_window, n_test_days, test_dates_nn, scaler
)
all_results["PatchTST"] = {"pred_monthly": ptst_pred_monthly, "true_monthly": true_monthly_dew}

ptst_mae = mean_absolute_error(true_monthly_dew, ptst_pred_monthly)
ptst_rmse = np.sqrt(mean_squared_error(true_monthly_dew, ptst_pred_monthly))
ptst_r2 = r2_score(true_monthly_dew, ptst_pred_monthly)
print(f"\nPatchTST Monthly DewPoint — MAE: {ptst_mae:.3f} | RMSE: {ptst_rmse:.3f} | R2: {ptst_r2:.4f}")

## Baselines & Model Comparison
All predictions are in original DewPoint scale (seasonal component added back).

In [ ]:
# Baselines
train_monthly_base = train_daily.groupby(["Year", "Month"])["DewPoint"].mean().reset_index()
all_monthly = df.groupby(["Year", "Month"])["DewPoint"].mean().reset_index()

# Naive Seasonal: same month from previous year
naive_dew = []
for _, row in true_monthly.iterrows():
    prev = all_monthly[(all_monthly["Year"] == row["Year"] - 1) & (all_monthly["Month"] == row["Month"])]
    naive_dew.append(prev["DewPoint"].values[0])
naive_dew = np.array(naive_dew)

# Climatological Mean: training average per month
clim_dew = []
for _, row in true_monthly.iterrows():
    month_avg = train_monthly_base[train_monthly_base["Month"] == row["Month"]]["DewPoint"].mean()
    clim_dew.append(month_avg)
clim_dew = np.array(clim_dew)

all_results["Naive Seasonal"] = {"pred_monthly": naive_dew, "true_monthly": true_monthly_dew}
all_results["Climatological Mean"] = {"pred_monthly": clim_dew, "true_monthly": true_monthly_dew}

# Comparison table
print("=" * 75)
print(f"  TRUE FORECASTING on DESEASONALIZED data")
print(f"  Predictions converted back to original DewPoint scale")
print(f"  Evaluated on MONTHLY averages")
print("=" * 75)
print(f"{'Model':<25} {'MAE':>8} {'RMSE':>8} {'R2':>8}")
print("-" * 75)
for name in ["SARIMA", "LSTM", "PatchTST", "Naive Seasonal", "Climatological Mean"]:
    r = all_results[name]
    mae = mean_absolute_error(r["true_monthly"], r["pred_monthly"])
    rmse = np.sqrt(mean_squared_error(r["true_monthly"], r["pred_monthly"]))
    r2 = r2_score(r["true_monthly"], r["pred_monthly"])
    print(f"{name:<25} {mae:>8.3f} {rmse:>8.3f} {r2:>8.4f}")
print("=" * 75)

In [ ]:
# Time series comparison plot
test_dates_monthly = pd.to_datetime(true_monthly[["Year", "Month"]].assign(Day=1))
colors = {"SARIMA": "red", "LSTM": "green", "PatchTST": "orange"}

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(test_dates_monthly, true_monthly_dew, "k-", label="Actual", linewidth=2)
for name, color in colors.items():
    ax.plot(test_dates_monthly, all_results[name]["pred_monthly"], "--", color=color,
            label=name, linewidth=1.3, alpha=0.85)
ax.plot(test_dates_monthly, naive_dew, ":", color="gray", label="Naive Seasonal", alpha=0.6)
ax.set_title("Monthly Average DewPoint — Trained on Deseasonalized Data (2012-2021)\n"
             "(Seasonal component added back for evaluation)")
ax.set_ylabel("DewPoint (°C)")
ax.legend(ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: predicted vs actual
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, color) in zip(axes.flat, colors.items()):
    pred = all_results[name]["pred_monthly"]
    r2 = r2_score(true_monthly_dew, pred)
    ax.scatter(true_monthly_dew, pred, c=color, alpha=0.6, s=30, edgecolors="white", linewidth=0.5)
    lims = [min(true_monthly_dew.min(), pred.min()) - 1, max(true_monthly_dew.max(), pred.max()) + 1]
    ax.plot(lims, lims, "k--", alpha=0.4)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("Actual Monthly DewPoint (°C)")
    ax.set_ylabel("Predicted Monthly DewPoint (°C)")
    ax.set_title(f"{name} — R² = {r2:.4f}")
    ax.grid(True, alpha=0.3)

plt.suptitle("Trained on Deseasonalized Data (seasonal added back)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# NN training loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, tl, vl) in zip(axes, [("LSTM", lstm_train_losses, lstm_val_losses),
                                       ("PatchTST", ptst_train_losses, ptst_val_losses)]):
    ax.plot(tl, label="Train")
    ax.plot(vl, label="Val")
    ax.set_title(f"{name} — Training & Validation Loss (deseasonalized)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Monthly residual boxplot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
test_months = true_monthly["Month"].values
month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

for ax, (name, color) in zip(axes.flat, colors.items()):
    residuals = all_results[name]["pred_monthly"] - true_monthly_dew
    month_groups = [residuals[test_months == m] for m in range(1, 13)]
    bp = ax.boxplot(month_groups, labels=month_labels, patch_artist=True)
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_alpha(0.3)
    ax.axhline(y=0, color="k", linestyle="--", alpha=0.5)
    ax.set_title(f"{name} — Monthly Residuals")
    ax.set_ylabel("Predicted - Actual (°C)")
    ax.grid(True, alpha=0.3)

plt.suptitle("Residuals (Deseasonalized Training)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Export predictions and metrics
results_df = pd.DataFrame({
    "Year": true_monthly["Year"].values,
    "Month": true_monthly["Month"].values,
    "Actual_DewPoint": true_monthly_dew,
})
for name in ["SARIMA", "LSTM", "PatchTST", "Naive Seasonal", "Climatological Mean"]:
    col = name.replace(" ", "_")
    results_df[f"Pred_{col}"] = all_results[name]["pred_monthly"]
results_df.to_csv("all_models_predictions.csv", index=False)

rows = []
for name in ["SARIMA", "LSTM", "PatchTST", "Naive Seasonal", "Climatological Mean"]:
    r = all_results[name]
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(r["true_monthly"], r["pred_monthly"]),
        "RMSE": np.sqrt(mean_squared_error(r["true_monthly"], r["pred_monthly"])),
        "R2": r2_score(r["true_monthly"], r["pred_monthly"]),
    })
metrics_df = pd.DataFrame(rows)
metrics_df.to_csv("all_models_metrics.csv", index=False)
print("Exported: all_models_predictions.csv, all_models_metrics.csv")
metrics_df